# Agentic RAG evaluation demo (Google Colab)

This notebook walks through an end-to-end **agentic RAG evaluation demo** on Colab (T4 GPU):

1. Clone the `ocr-agentic-rag` repo and install minimal dependencies.
2. Download the **FinQA** and **TAT-QA** RAG datasets.
3. (Optional) Generate a small `first_5_rows.json` preview for FinQA.
4. Build a **FinQA retriever index** on GPU (or CPU fallback).
5. Configure `ANTHROPIC_API_KEY` from **Colab Secrets**.
6. Run **RAG evaluation** separately for **FinQA** and **TAT-QA**.
7. Display aggregate metrics and sample counts for each dataset.
8. Optionally zip and download the FinQA index bundle for local use.

**Colab tip:** Go to **Runtime → Change runtime type** and select a **T4 GPU** (or another GPU) before running the index-building and evaluation sections. The notebook assumes `ANTHROPIC_API_KEY` is provided via **Colab Secrets** and will be loaded later using `google.colab.userdata`.


## What "agentic" RAG means here

**Strict definition (research-y):** an **autonomous** agent that plans, calls tools, and executes multi-step workflows end-to-end without human approval on each step.

**Loose, job-description definition (this notebook):** an **orchestrated, multi-step, tool-using RAG pipeline** where the model plans, retrieves, sometimes reranks, and executes numerical programs, but a human still reviews and signs off on results.

- **Normal RAG:**
  - Encode query → retrieve top-k chunks → generate a single-shot answer.
- **Agentic RAG (here):**
  - Plan query handling steps.
  - Retrieve with **corpus scoping** (e.g., FinQA vs TAT-QA corpora).
  - Optionally **rerank** retrieved passages.
  - Generate answers with **numerical program hints + execution**, then surface everything for human review.


## 1. Clone repo and install dependencies


In [ ]:
REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
REPO_DIR = "ocr-agentic-rag"

import os
os.environ["COLAB_GPU"] = os.environ.get("COLAB_GPU", "1")

if os.path.isdir(REPO_DIR):
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull")
else:
    get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)

!pip install -q sentence-transformers faiss-cpu rank_bm25 llama-index-core anthropic langgraph pandas


## 2. Download RAG datasets (FinQA + TAT-QA)

This step uses `scripts/download_rag_datasets.py` to fetch the FinQA and TAT-QA datasets into the locations expected by the evaluation pipeline and dataset adapters.

For example, to download only FinQA you could run:

```bash
!python scripts/download_rag_datasets.py --datasets finqa
```

In the cell below we download **both** FinQA and TAT-QA.


In [ ]:
!python scripts/download_rag_datasets.py --datasets finqa tatqa


## 3. Generate first_5_rows for FinQA (optional)

This is a small convenience helper that writes out the first 5 rows of the FinQA training set to `data/rag/FinQA/train/first_5_rows.json` so you can quickly inspect the schema and example questions.


In [ ]:
import json
from pathlib import Path

train_dir = Path("data/rag/FinQA/train")
train_qa_path = train_dir / "train_qa.json"
first_5_path = train_dir / "first_5_rows.json"

if train_qa_path.exists():
    with open(train_qa_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    rows = data[:5] if isinstance(data, list) else []
    with open(first_5_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Wrote {first_5_path} ({len(rows)} rows).")
else:
    print("train_qa.json not found; run section 2 first.")


## 4. Build FinQA retriever index (GPU on Colab)

We now build a **retriever index** over the FinQA corpus using the `HybridRetriever`.

- On Colab, we default to **GPU** (`COLAB_GPU=1`).
- For local or CPU-only runs, you can invoke the same script manually with the `--cpu` flag.

The script `scripts/build_finqa_embeddings_colab.py` is responsible for:
- Chunking the FinQA corpus using the same logic as `eval_runner._build_finqa_corpus_chunks`.
- Initializing `HybridRetriever(embedding_model="BAAI/bge-m3", device="cuda" or "cpu")`.
- Calling `retriever.build_index(chunks, batch_size=args.batch_size)`.
- Saving the index bundle via `retriever.save_index_bundle(output_dir, embedding_model="BAAI/bge-m3")`.


In [ ]:
import os

# Smaller, configurable batch size to reduce GPU memory usage
batch_size = os.environ.get("FINQA_EMBED_BATCH_SIZE", "64")

use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
if use_gpu:
    !python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --batch_size {batch_size}
else:
    !python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --cpu


## 5. Set API key (Colab Secrets)

The evaluation pipeline uses **Anthropic Claude** via the `ANTHROPIC_API_KEY` environment variable.

In Colab:

1. Go to **Settings → Secrets** (or use the key icon in the left sidebar).
2. Add a secret named `ANTHROPIC_API_KEY` with your API key value.
3. The cell below reads that secret using `google.colab.userdata` and sets `os.environ["ANTHROPIC_API_KEY"]`.


In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")


## 6a. Run RAG evaluation: FinQA

This runs the **agentic RAG** pipeline on the **FinQA** dataset only.

- `eval_runner.py` is invoked with `--category rag` and `--dataset FinQA`.
- Internally, `run_model(..., category="rag", dataset_name="FinQA")` should:
  - Use the `FinQAAdapter`.
  - Use RAG with **corpus scoping** (`corpus_id`) and the agentic orchestrator.
- Proof artifacts and metrics for FinQA are written under `data/proof/rag/finqa/`.


In [ ]:
!python eval_runner.py --category rag --max_split 1 --max_category 1 --dataset FinQA --debug --export_predictions_txt


## 6b. Run RAG evaluation: TAT-QA

This cell mirrors the FinQA run, but for **TAT-QA**:

- Uses `dataset_name="TATQA"` and the corresponding `TATQAAdapter`.
- Writes proof outputs to `data/proof/rag/tatqa/...`.


In [ ]:
!python eval_runner.py --category rag --max_split 1 --max_category 1 --dataset TATQA --debug --export_predictions_txt


## 7. Display results: FinQA

This section reads **FinQA** metrics and per-sample outputs from `data/proof/rag/finqa/` and displays:

- A small table of weighted-average metrics (if available).
- The number of samples evaluated per split.


In [ ]:
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
    import pandas as pd
    _use_display = True
except ImportError:
    _use_display = False

proof_dir = Path("data/proof/rag")
dataset_name = "finqa"
dataset_dir = proof_dir / dataset_name

if not dataset_dir.exists():
    print(f"No results for {dataset_name}. Run section 6a first.")
else:
    if _use_display:
        display(Markdown("### FinQA"))
    weighted = dataset_dir / f"{dataset_name}_weighted_avg.json"
    if weighted.exists():
        with open(weighted) as f:
            m = json.load(f)
        if _use_display:
            rows = [
                (k, f"{v:.4f}" if isinstance(v, (int, float)) and v is not None else str(v))
                for k, v in sorted(m.items())
            ]
            display(pd.DataFrame(rows, columns=["Metric", "Value"]))
        else:
            for k, v in sorted(m.items()):
                print(f"  {k}: {v}")
    for split_dir in dataset_dir.iterdir():
        if split_dir.is_dir():
            for f in split_dir.glob("*_per_sample_*.json"):
                with open(f) as fp:
                    n = len(json.load(fp))
                print(f"Samples evaluated: {n} ({split_dir.name})")
                break
            break


## 8. Display results: TAT-QA

Similar to the FinQA display cell, this section reads **TAT-QA** metrics and per-sample outputs from `data/proof/rag/tatqa/`.


In [ ]:
from pathlib import Path
import json

proof_dir = Path("data/proof/rag")
try:
    _use_display
except NameError:
    try:
        from IPython.display import Markdown, display
        import pandas as pd
        _use_display = True
    except ImportError:
        _use_display = False

dataset_name = "tatqa"
dataset_dir = proof_dir / dataset_name

if not dataset_dir.exists():
    print(f"No results for {dataset_name}. Run section 6b first.")
else:
    if _use_display:
        display(Markdown("### TAT-QA"))
    weighted = dataset_dir / f"{dataset_name}_weighted_avg.json"
    if weighted.exists():
        with open(weighted) as f:
            m = json.load(f)
        if _use_display:
            rows = [
                (k, f"{v:.4f}" if isinstance(v, (int, float)) and v is not None else str(v))
                for k, v in sorted(m.items())
            ]
            display(pd.DataFrame(rows, columns=["Metric", "Value"]))
        else:
            for k, v in sorted(m.items()):
                print(f"  {k}: {v}")
    for split_dir in dataset_dir.iterdir():
        if split_dir.is_dir():
            for f in split_dir.glob("*_per_sample_*.json"):
                with open(f) as fp:
                    n = len(json.load(fp))
                print(f"Samples evaluated: {n} ({split_dir.name})")
                break
            break


## 9. (Optional) Download index bundle for local use

If you want to reuse the FinQA retriever index on your local machine, you can zip it up and download it from Colab.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

src = Path("data/rag/FinQA/train/finqa_retriever_index")
if src.exists():
    zip_path = Path("finqa_retriever_index.zip")
    shutil.make_archive(zip_path.with_suffix(""), "zip", src.parent, src.name)
    files.download(str(zip_path))
    print("Downloaded. Unzip into data/rag/FinQA/train/ locally.")
else:
    print("Index not found; run section 4 first.")
